# First basket: is there an edge?

Library: `nba/first_basket.py` (functions below). Spec: project doc `claude/first-basket-edge-test.md`.

**Model.** P(player scores the game's first FG) = P(team scores first) × P(he is the team's first scorer).
- Share: conditional logit over each team's starters on log-share features (prior Q1 FG points, early FGA, own team-first history, FG%), all from games strictly before the date.
- Team: logistic on home + prior rates of scoring / allowing the first FG.

**Order (cheapest first, each step can stop the project)**
0. Market only: overround, de-vigged calibration, flat ROI by price bucket. No model needed.
1. Labels only: does the share model beat a no-fit heuristic and a uniform 1/5? Uses every game since 2022-23, no odds.
2. Edge: model vs de-vigged close. Paired log loss, blend weights (does the model add information the price lacks?), and a walk-forward bet replay.

**Grading.** `game_first` / `team_first` = first made field goal. DraftKings and BetMGM grade on that; FanDuel grades on first score including FTs (~9% of games differ), so FanDuel is left out by default.

**Pre-registered split (decide before looking at step 2 results)**
| Market | Develop on | Test once, after locking features and thresholds |
|---|---|---|
| Game first basket | 2024-25 | 2025-26 + 2026-27 |
| First team basket | 2025-26 Oct–Dec | 2026-01-01 onward |

**Decision rule.** Edge = blend `w_model` z > 2 on the test set **and** the walk-forward blend replay at the chosen threshold has a positive ROI with its 90% interval above −5%. Model-only ROI isn't enough: at ~+650 a season of bets has an ROI SE of ~5–8 pts (see the power table).

## Setup
Run from the repo root. Restore a fresh `nba.db` first (`git-workflow.md`): `Starter` was repaired Nov 8 and the model selects on it.

In [ ]:
import os, sys, sqlite3
os.chdir(os.path.expanduser('~/code/bets-dev'))
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nba import first_basket as fb

pd.set_option('display.width', 200, 'display.max_columns', 40, 'display.float_format', '{:.4f}'.format)
conn = sqlite3.connect(fb.DB_PATH)
print(fb.DB_PATH)

In [ ]:
# Starter sanity check: ~5 starters per team-game, every season. A stale DB shows ~0 before 2025-26.
chk = pd.read_sql("""select substr(game_date,1,7) as month, count(distinct game_id || team_id) as team_games,
                     sum(Starter) * 1.0 / count(distinct game_id || team_id) as starters_per_team_game
                     from plyrLogs where game_id like '002%' and min > 0 and game_date >= '2022-10-01'
                     group by 1 order by 1""", conn)
print(chk.iloc[::3].to_string(index=False))
assert chk.starters_per_team_game.between(4.8, 5.2).mean() > .95, 'Starter not repaired in this DB: restore a fresh dump'

## Player aliases for the extra store
The loader uses the extra store's `player_alias`, then the core store's. Build the extra one once per season (no API calls).

In [ ]:
from nba.NBAodds import oddsHistory
oh = oddsHistory(odds_dir=fb.EXTRA_DIR)
for s in ['2023-24', '2024-25', '2025-26', '2026-27']:
    cov, unmatched, fuzzy = oh.build_player_alias(s, min_coverage=0)
    print('{}: {:.2%} coverage, {} unmatched, {} fuzzy to review'.format(s, cov, len(unmatched), len(fuzzy)))

## EDA: labels (no odds)
Hypothesis generation only; nothing here is a conclusion.

In [ ]:
out = fb.load_outcomes(conn)
feat = fb.build_features(out)
teams = fb.build_team_frame(out)
out['season'] = np.where(out.game_date.dt.month >= 8, out.game_date.dt.year, out.game_date.dt.year - 1)
g = out.drop_duplicates('game_id')
print('games', len(g), '| valid_game share {:.3f}'.format(g.valid_game.mean()))
print(out.drop_duplicates(['game_id', 'team_id']).groupby('season').valid_team.mean().rename('valid_team share'))

In [ ]:
# Base rates: home team scores first, bench player is his team's first scorer
t = teams[teams.valid_game & (teams.home == 1)]
print('home scores first: {:.3f} (n={})'.format(t.scored_first.mean(), len(t)))
v = out[out.valid_team]
bench = v[v.team_first == 1].groupby('season').starter.apply(lambda s: 1 - s.mean())
print('bench first-scorer rate by season:'); print(bench)

In [ ]:
# Concentration: who scores first, by the starter's rank in prior Q1 FG points on his team
f = feat[feat.valid_team].copy()
f['q1_rank'] = f.groupby(['game_id', 'team_id']).q1_share.rank(ascending=False, method='first')
r = f.groupby('q1_rank').agg(team_first_rate=('team_first', 'mean'), mean_q1_share=('q1_share', 'mean'), n=('team_first', 'size'))
print(r)   # if team_first_rate tracks mean_q1_share, Q1 share is roughly calibrated as a raw probability

In [ ]:
# Is a player's first-basket rate persistent year to year? (skill vs noise)
s = out[(out.starter == 1) & out.valid_team].groupby(['player_id', 'season']).agg(k=('team_first', 'sum'), n=('team_first', 'size')).reset_index()
s = s[s.n >= 40]
s['rate'] = s.k / s.n
pr = s.merge(s.assign(season=s.season + 1), on=['player_id', 'season'], suffixes=('', '_prev'))
print('players with 40+ starts in consecutive seasons:', len(pr))
print('corr(rate, prior-season rate) = {:.3f}'.format(pr.rate.corr(pr.rate_prev)))
# binomial noise alone at these n caps the possible correlation; compare with the split-half below

In [ ]:
# Split-half reliability within a season (odd vs even starts): the ceiling for any persistent signal
o = out[(out.starter == 1) & out.valid_team].sort_values('game_date').copy()
o['half'] = o.groupby(['player_id', 'season']).cumcount() % 2
h = o.groupby(['player_id', 'season', 'half']).team_first.agg(['mean', 'size']).unstack('half')
h = h[(h[('size', 0)] >= 20) & (h[('size', 1)] >= 20)]
print('split-half corr = {:.3f} (n={})'.format(h[('mean', 0)].corr(h[('mean', 1)]), len(h)))

In [ ]:
# Top team-first scorers, 2025-26 starts (does the list look like usage leaders or like bigs who win the tip?)
names = pd.read_sql('select player_id, name from players', conn).astype({'player_id': str})
top = s[s.season == 2025].merge(names, on='player_id').sort_values('rate', ascending=False).head(20)
print(top[['name', 'n', 'k', 'rate']].to_string(index=False))

## Step 1: share model on labels
Walk-forward monthly refits from 2022-23. Negative `delta_vs_heuristic_x1000` = better than p ∝ prior Q1 FG points. Add one feature set at a time.

In [ ]:
lab = fb.run_labels(conn, '2023-10-01', '2026-04-15')
for k in ['share_models', 'team_first_model', 'bench_rate', 'factorization_check', 'coef_full']:
    print('\n==', k); print(lab[k].to_string(index=False))

Reading it:
- `skill_vs_uniform` near 0 = the share model barely beats 1/5. Then the first-team-basket market is a vig fight with no information edge.
- `team_first_model` delta vs coin flip near 0 is expected without tip-off data. Jump-ball winners from PlayByPlayV3 are the obvious next feature.
- `factorization_check`: a significant `log_q1_share` slope means p_game = p_team × p_share is biased for high-share players.

## Step 0: market only (development seasons)

In [ ]:
DEV = {'game': ['2024-25'], 'team': ['2025-26']}
mk = {}
for market, seasons in DEV.items():
    mk[market], _ = fb.run_market(conn, seasons, market)
    for k in ['coverage', 'book_summary', 'price_bucket_roi', 'calibration_book']:
        print('\n== {} {}'.format(market, k)); print(mk[market][k].to_string(index=False))
print(fb.roi_power())

In [ ]:
def reliability(tbl, title):
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.plot([0, .4], [0, .4], color='grey', lw=1)
    se = np.sqrt(tbl.p_mean * (1 - tbl.p_mean) / tbl.n)
    ax.errorbar(tbl.p_mean, tbl.actual, yerr=1.64 * se, fmt='o')
    ax.set_xlabel('predicted'); ax.set_ylabel('actual'); ax.set_title(title)
    plt.show()

for market in DEV:
    reliability(mk[market]['calibration_book'], '{}: de-vigged book (power)'.format(market))

A bucket with `roi_lo90` above 0 in `price_bucket_roi` is a model-free edge (e.g. longshot pricing). It still needs the test split to confirm.

## Step 2: model vs book (development seasons only)

In [ ]:
ed = {}
for market, seasons in DEV.items():
    ed[market] = fb.run_edge(conn, seasons, market)
    for k in ['log_loss_vs_book', 'blend_weights', 'bets_shop', 'bets_by_book']:
        print('\n== {} {}'.format(market, k)); print(ed[market][k].to_string(index=False))

In [ ]:
for market in DEV:
    reliability(ed[market]['calibration_model'], '{}: model (walk-forward)'.format(market))

In [ ]:
# Where does the model disagree with the book, and who's right?  Log ratio buckets on the eval frame.
e = ed['game']['eval_frame'].drop_duplicates(['game_id', 'player_id']).copy()
e['lr'] = np.log(e.p_model / e.p_book)
e['b'] = pd.qcut(e.lr, 10)
print(e.groupby('b', observed=True).agg(n=('y', 'size'), p_model=('p_model', 'mean'), p_book=('p_book', 'mean'), actual=('y', 'mean')))

Reading it:
- `blend_weights`: `w_model` with z > 2 means the model carries information the close doesn't. `w_book` ≈ 1 and `w_model` ≈ 0 means the market already has everything the model has.
- `log_loss_vs_book`: `model_minus_book` < 0 means the model alone beats the book. Unlikely; the blend is the realistic target.
- `bets_shop`: compare `wins` with `exp_wins_p` (claimed) and `exp_wins_price` (break-even). The winner's curse from threes applies: the claimed EV will overstate.

Lock the feature set, de-vig method and EV threshold here, then run the test cell **once**.

## Test (run once after locking)

In [ ]:
LOCKED = {'features': fb.SHARE_FEATURES, 'devig': 'power', 'threshold': 0.10}
TEST = {'game': ['2025-26', '2026-27'], 'team': ['2025-26', '2026-27']}
# te = {}
# for market, seasons in TEST.items():
#     te[market] = fb.run_edge(conn, seasons, market, LOCKED['devig'], features=LOCKED['features'])
#     ev = te[market]['eval_frame']
#     if market == 'team':
#         ev = ev[ev.game_date >= '2026-01-01']          # team market test window
#     print(market); print(fb.simulate_bets(ev, 'p_blend', (LOCKED['threshold'],)).to_string(index=False))
#     print(te[market]['blend_weights'].to_string(index=False))